# Phase 4 : Intelligence Artificielle Generative (LLM)
Dans ce notebook, nous comparons deux approches modernes de NLP :
1. **Zero-Shot Classification** (avec un modele Encoder type BERT/BART) : On classe sans entrainement.
2. **Few-Shot Learning** (avec un modele Decoder type GPT) : On donne des exemples a l'IA pour qu'elle complete la tache.

In [ ]:
from transformers import pipeline, set_seed
import pandas as pd
import os
import sys
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample

# Pour la reproductibilite des resultats GPT
set_seed(42)

## Etape 1 : Chargement d'un avis pour le test

In [ ]:
# On recupere un avis negatif aleatoire pour tester nos deux IA
df = load_yelp_sample('../data/raw/review.json', n_rows=1000)
bad_review = df[df['stars'] == 1].iloc[0]['text']
print(f'--- Texte a analyser ---')
print(f'{bad_review[:200]}...')

Chargement de 1000 lignes depuis ../data/raw/review.json...
--- Texte a analyser ---
I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) and was told they're too busy to do it. Really? The place is maybe half full at best. Does your dick...


## Approche 1 : Zero-Shot Classification (Encoder)
Utilisation de `distilbart` (modele type BERT) qui comprend le sens des categories sans exemples.

In [ ]:
# On definit des categories arbitraires
classifier = pipeline('zero-shot-classification', model='valhalla/distilbart-mnli-12-3')
labels = ['food quality', 'service', 'price', 'ambiance']

res = classifier(bad_review, labels)

print(f'Resultat Zero-Shot : Le probleme vient de : {res["labels"][0].upper()}')
print(f'Indice de confiance : {res["scores"][0]:.2%}')

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Resultat Zero-Shot : Le probleme vient de : SERVICE
Indice de confiance : 43.93%


## Approche 2 : Few-Shot Learning (Decoder)
C'est la methode demandee pour les LLM generatifs (comme ChatGPT ou DeepSeek).
On utilise ici **GPT-2** (un Decoder pur) et on lui fournit un prompt contenant des exemples (Few-Shot) pour qu'il comprenne la logique.

In [ ]:
from transformers import pipeline
import torch

# On utilise la version "Distill-Qwen-1.5B" de DeepSeek.
# C'est un modele recent (2025) qui est tres performant pour le raisonnement
# mais qui reste assez leger (environ 3 Go de RAM) pour tourner sur un PC portable.
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

print(f"Chargement du modele {model_name} en cours...")
print("Cela peut prendre quelques minutes selon la connexion internet (telechargement initial).")

# On initialise le pipeline de generation de texte
# On ne precise pas device_map="auto" pour eviter les bugs si pas de GPU Nvidia,
# le modele tournera sur le processeur (CPU) par defaut.
generator = pipeline(
    "text-generation",
    model=model_name,
    torch_dtype=torch.float32  # Force l'utilisation standard pour la compatibilite
)

print("Modele DeepSeek charge et pret a l'emploi.")

Chargement du modele deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B en cours...
Cela peut prendre quelques minutes selon la connexion internet (telechargement initial).


config.json:   0%|          | 0.00/679 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

In [ ]:
def classifier_avis_few_shot(avis_client):
    # Definition du Prompt (Consigne + Exemples)
    # DeepSeek comprend tres bien les instructions, on lui donne le contexte
    # et quelques exemples pour qu'il formate sa reponse exactement comme on veut.
    prompt = f"""Tu es un expert en analyse de sentiments. Classe l'avis suivant en POSITIF ou NEGATIF.
    
Exemple 1:
Avis: "Le serveur etait tres gentil et le plat delicieux."
Sentiment: POSITIF

Exemple 2:
Avis: "Trop d'attente, c'est inacceptable pour ce prix."
Sentiment: NEGATIVE

Exemple 3:
Avis: "Super ambiance, je recommande !"
Sentiment: POSITIF

Exemple 4:
Avis: "C'etait sale et bruyant."
Sentiment: NEGATIF

Tache a realiser:
Avis: "{avis_client}"
Sentiment:"""

    # Generation de la reponse
    # max_new_tokens=5 suffit car on attend juste un mot (POSITIF ou NEGATIF)
    # temperature=0.1 rend le modele tres deterministe (moins "creatif", plus rigoureux)
    resultat = generator(
        prompt, 
        max_new_tokens=10, 
        temperature=0.1,
        return_full_text=False, # On ne veut que la reponse generee, pas le prompt entier
        pad_token_id=generator.tokenizer.eos_token_id
    )
    
    # On recupere le texte genere et on nettoie les espaces
    reponse_brute = resultat[0]['generated_text'].strip()
    
    # On garde juste le premier mot au cas ou le modele bavarde un peu
    reponse_nettoyee = reponse_brute.split('\n')[0]
    
    return reponse_nettoyee

# Test rapide pour verifier que ca marche
print("--- Test DeepSeek Few-Shot ---")
avis_test = "La nourriture est bonne mais le service est catastrophique."
print(f"Avis : {avis_test}")
print(f"Resultat : {classifier_avis_few_shot(avis_test)}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=3) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Test du LLM (Decoder) ---
Avis : I am a long term frequent customer of this establishment. I just went in to order take out (3 apps) ...
Classification par GPT-2 : NEGATIVE
